In [ ]:
# Title description
# 2.Predict stock price by RNN
# we will use a simple RNN model to predict the next day's stock price based on historical prices.
# Example:
# Input: historical_prices = [0, 0] # yesterday and today
# Output: predicted_next_price = 0 # tomorrow's predicted price
# Constraints:
# 1 <= len(historical_prices) <= 10^4
# 0 <= historical_prices[i] <= 10^6

import random

class RNN:
    def __init__(self):
        # define the structure of the RNN
        self.w1 = 0
        self.w2 = 0
        self.w3 = 0
        self.b1 = 0
        self.b2 = 0
    
    def compute_gradients(self, historical_price: list[float], target: float): 
        def rule(x: float) -> float:
            return max(0, x)

        def rule_derivative(d: float) -> float:
            return 1.0 if d > 0 else 0.0

        x_step = []
        z_step = []
        y_step = []

        # forward loss
        y_prev = 0.0
        for x in historical_price:
            z = (
                self.w1 * x
                + self.b1
                + self.w2 * y_prev
            )
            y = rule(z)
            x_step.append(x)
            z_step.append(z)
            y_step.append(y)
            y_prev = y

        # prediction
        prediction = self.w3 * y_step[-1] + self.b2

        # loss
        loss = 0.5 * (target - prediction) ** 2
        # gradients
        dw1 = 0.0
        dw2 = 0.0
        dw3 = 0.0
        db1 = 0.0
        db2 = 0.0

        # output layer gradient
        # dw3 = loss * -1 * dpre/dw3 = loss * -1 * y_step[-1]
        dw3 = -loss * y_step[-1]
        db2 = -loss

        # backpropagation through time (BPTT) for hidden layer
        for t in reversed(range(len(historical_price))):
            # dw2 from where?
            # dloss/dw2 = dloss/dprediction * dprediction/dy * dy/dz * dz/dw2
            # because y_step contains the activations for each time step
            dw2 = -loss * self.w3 * rule_derivative(z_step[t]) * y_step[t-1] if t > 0 else -loss * self.w3 * rule_derivative(z_step[t]) * 0.0
            dw1 = -loss * self.w3 * rule_derivative(z_step[t]) * x_step[t]
            db1 = -loss * self.w3 * rule_derivative(z_step[t])
            # accumulate gradients over time
            dw2 += dw2
            dw1 += dw1
            db1 += db1

        return dw1, dw2, dw3, db1, db2
    def gradient_update(self, historical_price: list[float], target: float, learning_rate: float = 0.01):
        dw1, dw2, dw3, db1, db2 = self.compute_gradients(historical_price, target)
        self.w1 -= learning_rate * dw1
        self.w2 -= learning_rate * dw2
        self.w3 -= learning_rate * dw3
        self.b1 -= learning_rate * db1
        self.b2 -= learning_rate * db2

    
RNN = RNN()
example_1 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # Example input
target = 11
iteration = 10000  # Number of iterations for gradient updates
for _ in range(iteration):
    RNN.gradient_update(example_1, target)
    print("Weights and biases:", RNN.w1, RNN.w2, RNN.w3, RNN.b1, RNN.b2,   
          "prediction:", RNN.w3 * max(0, RNN.w1 * example_1[-1] + RNN.b1) + RNN.b2)